1. 문서의 내용을 읽는다
2. 문서를 쪼갠다
    - 토큰수 초과로 답변을 생성하지 못할 수 있고
    - 문서가 길면 (인풋이 길면) 답변 생성이 오래걸림
3. 임베딩 -> 백터 데이터베이스에 저장
4. 질문이 있을 때, 백터 데이터베이스에 유사도 검색
5. 유사도 검색으로 가져온 문서를 LLM에 질문과 같이 전달_dev test

In [1]:
%pip install python-dotenv langchain langchain-openai langchain-community langchain-text-splitters docx2txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 24.0 -> 25.0.1
[notice] To update, run: c:\Users\softone\.pyenv\pyenv-win\versions\3.12.3\python.exe -m pip install --upgrade pip


In [3]:
from langchain_community.document_loaders import Docx2txtLoader

loader = Docx2txtLoader('./tax2.docx')
document = loader.load()

BadZipFile: File is not a zip file

In [20]:
#document
len(document)

1

In [ ]:
from langchain_community.document_loaders import Docx2txtLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1500,
    chunk_overlap=200,
)

loader = Docx2txtLoader('./tax2.docx')
document_list = loader.load_and_split(text_splitter=text_splitter)

In [ ]:
#%pip install --upgrade langchain_chroma

  Using cached langchain_chroma-0.2.2-py3-none-any.whl.metadata (1.3 kB)
  Using cached numpy-1.26.4-cp312-cp312-win_amd64.whl.metadata (61 kB)
  Using cached chromadb-0.6.3-py3-none-any.whl.metadata (6.8 kB)
  Using cached build-1.2.2.post1-py3-none-any.whl.metadata (6.5 kB)
  Using cached chroma_hnswlib-0.7.6.tar.gz (32 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached fastapi-0.115.11-py3-none-any.whl.metadata (27 kB)
  Using cached uvicorn-0.34.0-py3-none-any.whl.metadata (6.5 kB)
  Using cached posthog-3.20.0-py2.py3-none-any.whl.metadata (2.9 kB)
  Using cached onnxruntime-1.21.0-cp312-cp312-win_amd64.whl.metadata (4.9 kB)
  Using cached opentelemetry_api-1.31.0-py3-none-any.whl.metadat

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  You can safely remove it manually.
  You can safely remove it manually.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location

In [10]:
from dotenv import load_dotenv
from langchain_openai import OpenAIEmbeddings

load_dotenv()

embedding = OpenAIEmbeddings(model='text-embedding-3-large')

In [11]:
from langchain_chroma import Chroma

#database = Chroma.from_documents(documents=document_list,embedding=embedding)
#database = Chroma.from_documents(documents=document_list,embedding=embedding, collection_name='chroma-tax', persist_directory="./chroma")
database= Chroma(collection_name='chroma-tax', persist_directory="./chroma", embedding_function=embedding)

In [4]:
query = '연봉 5천만원인 직장인의 소득세는 얼마인가요?'
#query = '제가 제공한 문맥은 어떤 내용인가요? 일부를 제공하여 줄 수 있나요? 혹시 인코딩이 깨지거나 하지는 않았나요? 연봉이 5천만원인 직장인의 소득세를 계산할 수 없나요?'
#retrieved_docs= database.similarity_search(query, k=3)

In [7]:
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model='gpt-4o')


In [ ]:
#prompt = f"""[Identity]
#- 당신은 최고의 한국 소득세 전문가 입니다.
#- [Context]를 참고하여 사용자의 질문에 답해주세요.

#[Context]
#{retrieved_docs}

#Question : {query}
#"""

In [ ]:
#ai_message = llm.invoke(prompt)

In [ ]:
#ai_message.content
#%pip install -U langchain langchainhub --quiet

'연봉 5천만원인 직장인의 소득세를 계산하기 위해서는 몇 가지 추가적인 정보가 필요합니다. 여기에서는 기본적인 계산 방법으로 개략적인 소득세를 설명드리겠습니다.\n\n1. **근로소득공제**: 연봉 5천만원에 대해 근로소득공제를 적용합니다. 근로소득공제는 일반적으로 연봉에 따라 차등 적용됩니다.\n\n2. **과세표준**: 총 소득에서 근로소득공제를 차감한 금액이 과세표준이 됩니다.\n\n3. **세율 적용**: 과세표준에 따라 소득세율을 적용하여 기본세액을 계산합니다. 한국의 소득세율은 과세표준에 따라 점진적으로 증가하는 구조(누진세율)입니다.\n\n4. **세액공제**: 기본세액에서 인적공제, 특별세액공제 등 다양한 세액공제를 받게 됩니다.\n\n이 내용을 바탕으로 예시 계산을 해보겠습니다:\n\n- **근로소득공제**: 연봉 5천만원일 경우, 근로소득공제가 대략 1,100만원 정도로 예상됩니다.\n- **과세표준**: 5천만원 - 1,100만원 = 3,900만원\n- **세율 적용**: 과세표준 3,900만원에 대한 소득세율은 24%입니다. (한국의 과세표준이 4,600만원 이하는 15%이지만, 여기에 적용할 누진세가 포함되어 24%대를 적용합니다.)\n- **기본세액**: 3,900만원 * 24% = 약 936만원\n- **세액공제**: 이 후, 인적 공제나 다른 공제를 적용한 최종 세액이 정해집니다.\n\n정확한 금액은 개인의 상황에 따라(예: 부양가족, 특별세액공제, 보험료 납부 등) 다를 수 있으므로, 보다 정확한 계산을 위해 국세청의 소득세 계산기를 활용하거나 세무 전문가의 상담을 받는 것이 좋습니다.'

In [5]:
from langchain import hub

prompt = hub.pull("rlm/rag-prompt")

c:\Users\softone\.pyenv\pyenv-win\versions\3.12.3\Lib\site-packages\langsmith\client.py:253: LangSmithMissingAPIKeyWarning: API key must be provided when using hosted LangSmith API
  warnings.warn(


In [12]:
#prompt
from langchain.chains import RetrievalQA

qa_chain = RetrievalQA.from_chain_type(
    llm, 
    retriever=database.as_retriever(),
    chain_type_kwargs={"prompt": prompt}
)

In [13]:
#ai_message = qa_chain({"query": query})
# 강의에서는 위처럼 진행하지만 업데이트된 LangChain 문법은 `.invoke()` 활용을 권장
ai_message = qa_chain.invoke({"query": query})

In [14]:
ai_message

{'query': '연봉 5천만원인 직장인의 소득세는 얼마인가요?',
 'result': '죄송하지만 제공된 문맥에 연봉 5천만 원인 직장인의 소득세 계산에 대한 정보는 포함되어 있지 않습니다. 소득세는 개인의 소득과 해당 연도 세율에 따라 달라질 수 있으며, 구체적인 세액 계산은 국세청의 세금 계산기를 통해 확인하는 것이 가장 정확합니다.'}

In [ ]:
%pip install --upgrade --quiet langchain-pinecone langchain-openai langchain pinecone-notebooks